# 04. 시각화 6종
**목표**: 미션 4.6의 차트 6종을 하나씩 그려 보고, **어떤 질문에 어떤 차트인지** 익힙니다. (예상 6~8시간)

> 규칙: 모든 차트에 **제목, x축 레이블, y축 레이블** 을 넣습니다.

In [ ]:
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 한글 폰트
if platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
elif platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
else:
    plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

rng = np.random.default_rng(3)
n = 600
df = pd.DataFrame({
    "category": rng.choice(["상의", "하의", "신발", "잡화"], n, p=[0.4, 0.25, 0.2, 0.15]),
    "price": np.concatenate([rng.normal(50000, 12000, n - 8), rng.normal(300000, 20000, 8)]),
    "qty": rng.integers(1, 6, n),
    "order_date": pd.to_datetime("2025-01-01") + pd.to_timedelta(rng.integers(0, 365, n), unit="D"),
})
df["amount"] = df["price"] * df["qty"]
df.head()

## 1. 히스토그램 — "값이 어떻게 분포하는가?"
수치형 변수 하나의 분포. 막대 개수(`bins`)에 따라 모양이 달라지니 여러 값을 시도해 봅니다.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df["price"], bins=30, color="steelblue", edgecolor="white")
ax.set_title("상품 가격 분포")
ax.set_xlabel("가격(원)")
ax.set_ylabel("상품 수")
plt.show()

## 2. 박스플롯 — "이상치가 있는가? 처리 전후는?"
상자 = Q1~Q3(IQR), 가운데 선 = 중앙값, 점 = 이상치입니다. 미션은 **처리 전후 비교**를 요구하므로 두 개를 나란히 그립니다.

In [ ]:
q1, q3 = df["price"].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
after = df["price"].clip(lower, upper)

fig, axes = plt.subplots(1, 2, figsize=(9, 4), sharey=True)
sns.boxplot(y=df["price"], ax=axes[0], color="lightcoral")
axes[0].set_title("이상치 처리 전")
axes[0].set_ylabel("가격(원)")
axes[0].set_xlabel("전체 상품")

sns.boxplot(y=after, ax=axes[1], color="lightgreen")
axes[1].set_title("이상치 처리 후 (클리핑)")
axes[1].set_ylabel("가격(원)")
axes[1].set_xlabel("전체 상품")
plt.tight_layout()
plt.show()

## 3. 막대그래프 — "카테고리별 개수/크기는?"

In [ ]:
counts = df["category"].value_counts()

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(counts.index, counts.values, color="slateblue")
ax.set_title("카테고리별 주문 건수")
ax.set_xlabel("카테고리")
ax.set_ylabel("주문 건수")
for i, v in enumerate(counts.values):
    ax.text(i, v + 3, str(v), ha="center")   # 막대 위에 숫자 표시
plt.show()

## 4. 히트맵 — "변수끼리 얼마나 연관되는가?"
상관행렬을 색으로 표현합니다. (RFM 세그먼트 분포에도 같은 방식으로 씁니다.)

In [ ]:
corr = df[["price", "qty", "amount"]].corr()

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax)
ax.set_title("수치형 변수 상관행렬")
ax.set_xlabel("변수")
ax.set_ylabel("변수")
plt.show()

## 5. 산점도 — "두 변수가 함께 움직이는가?"

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(df["qty"], df["amount"], alpha=0.4, s=15)
ax.set_title("구매 수량과 주문 금액의 관계")
ax.set_xlabel("구매 수량(개)")
ax.set_ylabel("주문 금액(원)")
plt.show()

## 6. 라인차트 — "시간에 따라 어떻게 변하는가?"

In [ ]:
monthly = df.set_index("order_date")["amount"].resample("MS").sum()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(monthly.index, monthly.values, marker="o")
ax.set_title("월별 매출 추이")
ax.set_xlabel("월")
ax.set_ylabel("매출(원)")
ax.grid(alpha=0.3)
fig.autofmt_xdate()
plt.show()

## 차트 고르는 요령
| 질문 | 차트 |
|---|---|
| 분포가 어떤가 | 히스토그램 |
| 이상치/그룹 비교 | 박스플롯 |
| 범주별 크기 | 막대 |
| 변수 간 연관 한눈에 | 히트맵 |
| 두 변수 관계 | 산점도 |
| 시간 추세 | 라인 |

## 차트를 보고 해석 쓰기 (연습)
> 상품 가격 히스토그램은 4~6만 원 구간에 대부분 몰려 있고, 30만 원대에 소수의 고가 상품이 있다(8개). → 고가 상품은 별도 세그먼트로 관리하거나 이상치 처리 후 분석해야 한다.

### ✏️ 직접 해보기
카테고리별 **평균 주문 금액**을 막대그래프로 그리세요. (제목과 축 레이블 필수)

> 힌트: `df.groupby('category')['amount'].mean()`

In [ ]:
# 여기에 직접 작성해 보세요

**정답 예시** (먼저 직접 풀어본 뒤 확인하세요)

In [ ]:
avg = df.groupby("category")["amount"].mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(avg.index, avg.values, color="teal")
ax.set_title("카테고리별 평균 주문 금액")
ax.set_xlabel("카테고리")
ax.set_ylabel("평균 주문 금액(원)")
plt.show()

✅ 다음: `05_rfm_concept`